In [ ]:
import sys
from pathlib import Path
_here = Path.cwd().resolve()
for _p in [_here, _here / "notebooks", _here.parent]:
    if (_p / "_nb_common.py").exists() and str(_p) not in sys.path:
        sys.path.insert(0, str(_p))
        break
import _nb_common as nb
nb.style()
rng = nb.set_seed()
nb.env_report()


# Membership Inference Attack (MIA) Evaluation

**Purpose.** Evaluate empirical membership leakage of a trained model using black-box confidence/loss signals and project MIA utilities. Results are **empirical attack outcomes, not formal privacy guarantees**.

## Learning objectives
- Understand black-box membership inference threat models.
- Use `insurance_fl.evaluation.mia` (`compute_features`, `tpr_at_fpr`) and interpret ROC-AUC, TPR@1%FPR, and advantage.
- Load a precomputed artifact when available, or run a light experiment from data.
- Visualize score separation, ROC/PR (including low-FPR view), and illustrative defense comparisons.

## Background and math

### Threat model (black-box)
| Attacker knowledge | Example capability | Typical signal |
|---|---|---|
| Model outputs only | Query API, observe probabilities/confidence | max prob, entropy, loss proxy |
| Shadow data distribution | Train shadow models mimicking target | attack classifier on shadow-derived features |
| Limited/unknown training set | No exact members known | relies on generalization gap |

*Meaning here:* we assume the attacker can obtain model outputs for chosen inputs and knows a similar data distribution (shadow setting), but not model parameters.

### Metrics
- ROC-AUC: overall separability of members vs non-members.
- TPR@FPR (e.g. 1%): attack success when false positives must be rare.
- Advantage: $\mathrm{Adv}=\max_t (\mathrm{TPR}(t)-\mathrm{FPR}(t))$.
*Meaning here:* AUC near 0.5 suggests weak separation by this attack; high TPR at low FPR indicates practically concerning leakage.

## Assumptions and limitations
- Empirical only (D-MIA-001 context): low attack success does **not** prove privacy; stronger attacks or different data may succeed.
- Light-mode experiments use small samples/models for runtime; results are illustrative.
- Any loaded JSON artifact is reported as **precomputed artifact** with its recorded metadata.


In [ ]:
from dataclasses import dataclass
@dataclass(frozen=True)
class Cfg:
    seed: int = nb.SEED
    n_rows: int = 1200 if nb.MODE == "light" else 4000
    test_size: float = 0.4
    epochs: int = 2 if nb.MODE == "light" else 8
    hidden: int = 32
cfg = Cfg(); cfg


In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.metrics import roc_curve, roc_auc_score, precision_recall_curve, auc
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from insurance_fl.evaluation.mia import compute_features, tpr_at_fpr

data_root = nb.require_data()
print('data_root:', data_root)

artifact_path = nb.REPORTS / 'mia_evaluation.json'
artifact = None
if artifact_path.exists():
    try:
        artifact = nb.load_artifact(artifact_path)
        print('[artifact loaded]', artifact_path)
        print(json.dumps(artifact, indent=2))
    except Exception as exc:
        print(f'[artifact skipped] {artifact_path}: {type(exc).__name__}: {exc}')
else:
    print('[artifact skipped] missing', artifact_path, '; run scripts/run_mia.py to generate it')


## Light experiment (computed if needed / for visualization)

In [ ]:
# Build a simple binary task (is_high_risk) with member/non-member split
clients = nb.load_clients(split='train', n_max=cfg.n_rows)
df = pd.concat([v.assign(client=k) for k, v in clients.items()], ignore_index=True)
target = 'is_high_risk' if 'is_high_risk' in df.columns else None
if target is None:
    raise RuntimeError('is_high_risk not found in data')
feat_cols = [c for c in df.select_dtypes(include=[np.number]).columns if c != target]
X = df[feat_cols].values
y = df[target].astype(int).values
Xm, Xnm, ym, ynm = train_test_split(
    X, y,
    test_size=cfg.test_size,
    random_state=nb.SEED,
    stratify=y if len(np.unique(y)) > 1 else None,
)
assert len(Xm) > 0 and len(Xnm) > 0
assert len(Xm) + len(Xnm) == len(X)
# target model (attacked): trained on members
clf = Pipeline([
    ('imp', SimpleImputer(strategy='median')),
    ('sc', StandardScaler()),
    ('lr', LogisticRegression(max_iter=500, random_state=nb.SEED)),
])
clf.fit(Xm, ym)
# black-box scores: probability of positive class and loss-like confidence
pm = clf.predict_proba(Xm)[:, 1]
pnm = clf.predict_proba(Xnm)[:, 1]
assert np.all((pm >= 0) & (pm <= 1)) and np.all((pnm >= 0) & (pnm <= 1))
# per-sample log loss as attack signal
eps = 1e-12
loss_m = -(ym * np.log(pm + eps) + (1 - ym) * np.log(1 - pm + eps))
loss_nm = -(ynm * np.log(pnm + eps) + (1 - ynm) * np.log(1 - pnm + eps))
# membership labels: 1=member, 0=non-member; attack score = higher member-likelihood
# use negative loss as membership score (members often have lower loss)
score_m = -loss_m
score_nm = -loss_nm
y_mem = np.concatenate([np.ones_like(score_m), np.zeros_like(score_nm)])
scores = np.concatenate([score_m, score_nm])
assert len(y_mem) == len(scores) == (len(Xm) + len(Xnm))
fpr, tpr, thr = roc_curve(y_mem, scores)
roc_auc = roc_auc_score(y_mem, scores) if len(np.unique(y_mem)) > 1 else float('nan')
adv = float(np.max(tpr - fpr)) if len(tpr) else float('nan')
tpr1 = float(tpr_at_fpr(y_mem, scores, 0.01))
print({
    'roc_auc': float(roc_auc) if np.isfinite(roc_auc) else None,
    'tpr_at_1pct_fpr': tpr1,
    'advantage': adv,
    'n_member': int(len(Xm)),
    'n_nonmember': int(len(Xnm)),
})
assert np.isnan(roc_auc) or (0.0 <= float(roc_auc) <= 1.0)
assert 0.0 <= tpr1 <= 1.0
assert np.isnan(adv) or (0.0 <= adv <= 1.0)


In [ ]:
# Try project compute_features/tpr_at_fpr on a torch-like wrapper if possible (best-effort)
torch = nb.optional('torch')
project_mia_note = None
proj_feats_shape = None
if torch is not None:
    try:
        # minimal torch model wrapper compatible with compute_features expectations
        class TorchLogReg(torch.nn.Module):
            def __init__(self, d):
                super().__init__()
                self.lin = torch.nn.Linear(d, 1)
            def forward(self, x):
                return self.lin(x)
        m = TorchLogReg(Xm.shape[1])
        # quick fit
        opt = torch.optim.Adam(m.parameters(), lr=1e-2)
        Xm_imp_full = SimpleImputer(strategy='median').fit_transform(Xm)
        Xt = torch.tensor(Xm_imp_full, dtype=torch.float32)
        yt = torch.tensor(ym.reshape(-1, 1), dtype=torch.float32)
        m.train()
        for _ in range(max(1, cfg.epochs * 20)):
            opt.zero_grad()
            loss = torch.nn.functional.binary_cross_entropy_with_logits(m(Xt), yt)
            loss.backward()
            opt.step()
        n_demo = min(200, len(Xm_imp_full))
        feats = compute_features(m, Xm_imp_full[:n_demo], ym[:n_demo], task_type='binary')
        proj_feats_shape = tuple(getattr(feats, 'shape', ())) or None
        project_mia_note = 'compute_features executed successfully (project API)'
    except Exception as e:
        project_mia_note = f'[fallback] project compute_features demo unavailable: {e}'
else:
    project_mia_note = '[fallback] torch not installed; project MIA torch feature demo skipped'
print(project_mia_note, proj_feats_shape)


## Figures: attack behavior

In [ ]:
# Figure 1: member vs non-member score histograms
fig, axes = plt.subplots(1,2, figsize=(11,4))
axes[0].hist(score_m, bins=30, alpha=0.5, density=True, label='member (-loss)')
axes[0].hist(score_nm, bins=30, alpha=0.5, density=True, label='non-member (-loss)')
axes[0].set_title('Membership score distributions'); axes[0].set_xlabel('Score (higher => more member-like)'); axes[0].set_ylabel('Density'); axes[0].legend()
axes[1].hist(pm, bins=30, alpha=0.5, density=True, label='member P(y=1)')
axes[1].hist(pnm, bins=30, alpha=0.5, density=True, label='non-member P(y=1)')
axes[1].set_title('Prediction confidence proxy'); axes[1].set_xlabel('Predicted probability'); axes[1].set_ylabel('Density'); axes[1].legend()
fig.tight_layout(); nb.save_fig(fig, 'membership_inference_score_hists'); plt.show()


In [ ]:
# Figure 2: ROC (linear and log-log)
fig, axes = plt.subplots(1,2, figsize=(11,4))
axes[0].plot(fpr, tpr, label=f'ROC AUC={roc_auc:.3f}')
axes[0].plot([0,1],[0,1],'k--', lw=1, label='chance')
axes[0].set_title('ROC curve'); axes[0].set_xlabel('False Positive Rate'); axes[0].set_ylabel('True Positive Rate'); axes[0].legend()
# log-log view (clip for log scale)
fpr_c = np.clip(fpr, 1e-4, 1.0); tpr_c = np.clip(tpr, 1e-4, 1.0)
axes[1].loglog(fpr_c, tpr_c, label=f'TPR@1%FPR={tpr1:.3f}')
axes[1].loglog([1e-4,1],[1e-4,1],'k--', lw=1)
axes[1].set_title('ROC (log-log, low-FPR focus)'); axes[1].set_xlabel('FPR (log)'); axes[1].set_ylabel('TPR (log)'); axes[1].legend()
fig.tight_layout(); nb.save_fig(fig, 'membership_inference_roc'); plt.show()


In [ ]:
# Figure 3: Precision-Recall
prec, rec, _ = precision_recall_curve(y_mem, scores)
pr_auc = auc(rec, prec)
fig, ax = plt.subplots(figsize=(6.5,4))
ax.plot(rec, prec, label=f'PR AUC={pr_auc:.3f}')
ax.set_title('Precision-Recall curve'); ax.set_xlabel('Recall (TPR)'); ax.set_ylabel('Precision'); ax.legend()
fig.tight_layout(); nb.save_fig(fig, 'membership_inference_pr'); plt.show()


**How to read this / what it shows.** PR curves are informative when classes (member/non-member evaluation sets) are imbalanced; precision indicates how often flagged members are true members.

In [ ]:
# Figure 4: advantage vs train/test gap and illustrative defenses
# gap proxy: member accuracy/confidence minus non-member
gap = float(np.mean(pm>0.5) - np.mean(pnm>0.5)) if len(pm) and len(pnm) else 0.0
defense_rows = [
    {'setting':'baseline', 'roc_auc': roc_auc, 'advantage': adv, 'gap_proxy': gap},
    {'setting':'stronger_reg (illustrative)', 'roc_auc': min(roc_auc, 0.55) if np.isfinite(roc_auc) else np.nan, 'advantage': min(adv, 0.10) if np.isfinite(adv) else np.nan, 'gap_proxy': gap*0.7},
    {'setting':'noise/DP (illustrative)', 'roc_auc': min(roc_auc, 0.53) if np.isfinite(roc_auc) else np.nan, 'advantage': min(adv, 0.08) if np.isfinite(adv) else np.nan, 'gap_proxy': gap*0.5},
]
def_df = pd.DataFrame(defense_rows)
fig, ax = plt.subplots(figsize=(7,4))
x = np.arange(len(def_df))
ax.bar(x-0.15, def_df['roc_auc'].fillna(0), width=0.3, label='ROC AUC')
ax.bar(x+0.15, def_df['advantage'].fillna(0), width=0.3, label='Advantage')
ax.set_xticks(x); ax.set_xticklabels(def_df['setting'], rotation=15)
ax.set_ylim(0,1); ax.set_title('Attack metrics and illustrative defenses (non-exhaustive)')
ax.legend()
fig.tight_layout(); nb.save_fig(fig, 'membership_inference_defenses'); plt.show()
def_df


**How to read this / what it shows.** Baseline metrics are computed from this run. Defense rows marked illustrative show expected direction (reduced gap/leakage) but are not measured deployments unless explicitly recomputed.

In [ ]:
# Figure: score-by-example scatter from the computed attack
idx_member = np.flatnonzero(y_mem == 1)
idx_nonmember = np.flatnonzero(y_mem == 0)
fig, ax = plt.subplots(figsize=(7,4))
ax.scatter(idx_member, scores[idx_member], s=12, alpha=0.55, label='member')
ax.scatter(idx_nonmember, scores[idx_nonmember], s=12, alpha=0.55, label='non-member')
ax.set_title('Attack scores by held-out example'); ax.set_xlabel('Example index'); ax.set_ylabel('Membership score'); ax.legend()
fig.tight_layout(); nb.save_fig(fig, 'membership_score_scatter'); plt.show()


**How to read this / what it shows.** Points show the attack score and known split label for each scored example. The index is only a plotting position; class overlap indicates the attack does not cleanly separate these samples.

## Quantitative table


In [ ]:
table = pd.DataFrame([{
    'source': 'precomputed_artifact+computed' if artifact is not None else 'computed',
    'artifact_auc': (artifact or {}).get('roc_auc') if isinstance(artifact, dict) else None,
    'artifact_tpr_at_1pct_fpr': (artifact or {}).get('tpr_at_1pct_fpr') if isinstance(artifact, dict) else None,
    'computed_auc': float(roc_auc) if np.isfinite(roc_auc) else None,
    'computed_tpr_at_1pct_fpr': float(tpr1),
    'computed_advantage': float(adv) if np.isfinite(adv) else None,
    'n_member': int(len(Xm)),
    'n_nonmember': int(len(Xnm)),
    'project_mia_api': project_mia_note,
}])
table


## Sensitivity / ablation
- Larger train/test gaps and overfitting typically increase attack advantage.
- Attack strength depends on features/signals, shadow model quality, and sample sizes.
- Re-run with `NB_MODE=full` and more epochs for tighter (but still empirical) estimates.

## Limitations and failure cases
- This is one attack family; adaptive or stronger attacks may perform better.
- AUC/TPR near random does **not** certify privacy (not a DP guarantee).
- Precomputed artifacts reflect the conditions recorded in the JSON; they are not re-validated here beyond loading.

## Conclusions and further experiments
- We reported empirical MIA metrics and visual diagnostics from real model outputs on bundled data.
- Combine with DP accounting/training controls for stronger protection arguments.
- Further work: multiple shadow models, per-client/per-record analysis, and stronger white-box/label-only attacks.

## References
- R. Shokri et al. "Membership Inference Attacks Against Machine Learning Models." *IEEE S&P*, 2017.
- S. Yeom et al. "Privacy Risk in Machine Learning: Analyzing the Connection to Overfitting." *CSF*, 2018.
- N. Carlini et al. "Membership Inference Attacks From First Principles." *IEEE S&P*, 2022.
